# 0.3 - CatBoost Modelling

Trains CatBoost on log1p(radon) with building-level train/validation subsets. Evaluation: 0.3.1.

## Setup

In [ ]:
import pickle

import catboost as cb
import numpy as np
import optuna
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupShuffleSplit

ROOT = '../../'
DIR = '../../models/'
RANDOM_SEED = 42
N_TRIALS = 50  # same in 0.3 and 0.5
FIXED_PARAMS = {'random_seed': RANDOM_SEED, 'used_ram_limit': '8gb', 'verbose': False}

## Data

In [ ]:
X_train = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-x-train.csv', index_col=0).replace([np.inf, -np.inf], np.nan)
y_train = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-y-train.csv', index_col=0).squeeze()
egid_train = pd.read_csv(f'{ROOT}data/final/dataset-v0.2-egid-train.csv', index_col=0)['EGID']

# Validation subset: 20% of the training buildings
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, val_idx = next(splitter.split(X_train, y_train, groups=egid_train.loc[X_train.index]))
X_train, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
y_train, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

# 0.2 pipeline, fitted on the training and validation subsets: used for the final model
with open(f'{DIR}0.2-encoding-pipeline.pkl', 'rb') as file:
    preprocessor = pickle.load(file)

# Same pipeline refitted on the training subset only: used for tuning and the validation predictions
train_preprocessor = clone(preprocessor).fit(X_train)
X_train_enc = train_preprocessor.transform(X_train)
X_val_enc = train_preprocessor.transform(X_val)

X_train.shape, X_val.shape

## Hyperparameter optimization

Trained on log1p(Bq/m3), scored with R² (log scale) on the validation subset.

In [ ]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 400),
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.2),
        'depth': trial.suggest_int('depth', 1, 12),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.2, 5),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.01, 0.1),
        'boosting_type': trial.suggest_categorical('boosting_type', ['Ordered', 'Plain']),
        'bootstrap_type': trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS']),
    }
    if params['bootstrap_type'] == 'Bayesian':
        params['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0, 10)
    elif params['bootstrap_type'] == 'Bernoulli':
        params['subsample'] = trial.suggest_float('subsample', 0.1, 1)

    model = cb.CatBoostRegressor(**params, **FIXED_PARAMS)
    model.fit(X_train_enc, np.log1p(y_train))
    return r2_score(np.log1p(y_val), model.predict(X_val_enc))

In [ ]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED), storage='sqlite:///db.sqlite3')
study.optimize(objective, n_trials=N_TRIALS)

best_params = study.best_params
print(f'Best validation R2 (log): {study.best_value:.3f}')
best_params 

## Validation predictions

Model trained on the training subset only; its validation predictions are used to choose the decision threshold in 0.3.1.

In [ ]:
val_model = cb.CatBoostRegressor(**best_params, **FIXED_PARAMS)
val_model.fit(X_train_enc, np.log1p(y_train))
pd.DataFrame({'observed': y_val, 'predicted': np.expm1(val_model.predict(X_val_enc))}).to_csv(f'{DIR}0.3-catboost-val-predictions.csv')

## Final model

Retrained on the training and validation subsets. Predicts log1p(Bq/m3): back-transform with `np.expm1`.

In [ ]:
model = cb.CatBoostRegressor(**best_params, **FIXED_PARAMS)
model.fit(preprocessor.transform(pd.concat([X_train, X_val])), np.log1p(pd.concat([y_train, y_val])))
model.save_model(f'{DIR}0.3-catboost-log-model.cbm')